# Tent reproduction (Student A) on Colab

Re-runs the three upstream configs (`source`, `norm`, `tent`) on both WRN architectures and two seeds, then builds the README-format tables, the severity-trend plot, the seed-variance summary, and a deviation report.

**Why the extra environment step:** the upstream pins are a Python-3.8 / 2020-era set (`torch==1.8.1`, and robustbench v0.1 drags in `numpy~=1.19.4`), which will not build on Colab's stock Python 3.11+. We create a throwaway Python-3.8 env with `uv` and install `requirements.txt` *unchanged* inside it, so the reproduction stays faithful. The one unavoidable deviation is `torch 1.8.1+cu111` (vs the authors' cu102) — `deviation_report.md` flags any resulting wobble.

**Paths used throughout** (edit `DRIVE_ROOT` / repo coordinates in the setup cell if yours differ):

| what | path |
|---|---|
| repo clone | `/content/FRMDL-Tent-Reproducibility` (branch `reproducibility`) |
| Python 3.8 venv | `/content/venv` |
| CIFAR-10-C + checkpoints | **Google Drive** `…/frmdl_tent/{data,ckpt}` (symlinked into `tent/`, so downloaded once) |
| logs + results | **Google Drive** `…/frmdl_tent/output_A` |

> Runtime: the full 12-run matrix is ~2–4 h on a T4. Logs stream to Drive and the run is resumable (`SKIP_EXISTING=1`), so a disconnect just means re-running the run cell. For a quick headline number first, use the *fast path* cell.

## 0. Confirm the GPU runtime
Runtime ▸ Change runtime type ▸ **T4 GPU**, then run:

In [ ]:
!nvidia-smi -L

## 1. Mount Drive and define paths
All other cells read these Python variables (IPython expands `$VAR` inside `!` shell lines).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# --- edit these if your repo/Drive layout differs -------------------------
GITHUB_USER = 'athxrva02'
GITHUB_REPO = 'FRMDL-Tent-Reproducibility'
BRANCH      = 'reproducibility'
DRIVE_ROOT  = '/content/drive/MyDrive/frmdl_tent'
# --------------------------------------------------------------------------

REPO_DIR  = f'/content/{GITHUB_REPO}'
TENT_DIR  = f'{REPO_DIR}/tent'
VENV      = '/content/venv'
OUT_ROOT  = f'{DRIVE_ROOT}/output_A'
DATA_DIR  = f'{DRIVE_ROOT}/data'
CKPT_DIR  = f'{DRIVE_ROOT}/ckpt'

for d in (OUT_ROOT, DATA_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)
print('paths ready:', OUT_ROOT)

## 2. Clone (or update) the repo
`repro_a/` must already be committed and pushed to the `reproducibility` branch. For a **private** repo, replace the URL with `https://<PAT>@github.com/...` using a read-scoped Personal Access Token.

In [ ]:
if not os.path.isdir(REPO_DIR):
    !git clone -b $BRANCH https://github.com/$GITHUB_USER/$GITHUB_REPO.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull --ff-only
!ls $TENT_DIR/repro_a

## 3. Persist data + checkpoints on Drive
Symlink `tent/data` and `tent/ckpt` to Drive so CIFAR-10-C (~2.9 GB) and the RobustBench checkpoints download **once** and survive session restarts. `cifar10c.py` uses the default `./data` / `./ckpt`, which now resolve to Drive.

> Prefer fast ephemeral storage instead? Skip this cell — the data re-downloads to `/content` each session (a few minutes), which avoids slower Drive FUSE reads.

In [ ]:
!ln -sfn $DATA_DIR $TENT_DIR/data
!ln -sfn $CKPT_DIR $TENT_DIR/ckpt
!ls -ld $TENT_DIR/data $TENT_DIR/ckpt

## 4. Build the Python 3.8 environment
`uv` fetches a standalone Python 3.8 and installs the upstream pins into `/content/venv`. `torch 1.8.1+cu111` satisfies the `==1.8.1` pin and runs on the T4. (~3–5 min.)

In [ ]:
!pip install -q uv
!uv venv --python 3.8 $VENV
!uv pip install --python $VENV/bin/python \
    torch==1.8.1+cu111 torchvision==0.9.1+cu111 \
    -f https://download.pytorch.org/whl/torch_stable.html
!uv pip install --python $VENV/bin/python -r $TENT_DIR/requirements.txt

Sanity-check that the 3.8 env sees the GPU:

In [ ]:
!$VENV/bin/python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"
# expect: 1.8.1+cu111 True Tesla T4

## 5. Smoke test (~2 min)
Downloads the checkpoint + CIFAR-10-C to Drive and runs one tiny eval. `run_all.sh` `cd`s into `tent/` itself, so it can be called by absolute path.

In [ ]:
!PY=$VENV/bin/python OUT_ROOT=$OUT_ROOT bash $TENT_DIR/repro_a/run_all.sh --smoke

## 6. Run the full 12-run matrix
2 archs × 3 methods × 2 seeds. Logs stream to Drive; `SKIP_EXISTING=1` makes it resumable — if Colab disconnects, just re-run this cell and finished runs are skipped.

In [ ]:
!PY=$VENV/bin/python OUT_ROOT=$OUT_ROOT SKIP_EXISTING=1 bash $TENT_DIR/repro_a/run_all.sh

### (Optional) Fast path — headline WRN-28-10 numbers only
Runs `source`/`norm`/`tent` on `Standard`, seed 1, to get the 18.6 figure before committing hours to the full matrix. The analysis cells work on this partial tree.

In [ ]:
%%bash
VENV=/content/venv
TENT=/content/FRMDL-Tent-Reproducibility/tent
OUT=/content/drive/MyDrive/frmdl_tent/output_A
cd "$TENT"
for m in source norm tent; do
  "$VENV/bin/python" cifar10c.py --cfg cfgs/$m.yaml \
    RNG_SEED 1 SAVE_DIR "$OUT/Standard/$m/seed1"
done

## 7. Build the deliverables
`parse_logs.py` is stdlib-only and `make_tables.py` needs only pandas + matplotlib (both preinstalled in Colab's base Python), so the analysis runs outside the 3.8 venv.

In [ ]:
!python $TENT_DIR/repro_a/parse_logs.py  --root $OUT_ROOT --out $OUT_ROOT/results.csv
!python $TENT_DIR/repro_a/make_tables.py --csv  $OUT_ROOT/results.csv --out $OUT_ROOT

## 8. View the results
**Reproduction sanity:** WRN-28-10 `tent` severity-5 mean should be **18.6 ± ~1pp**; >2pp is flagged automatically (likely cause: cu111 vs the authors' cuDNN/torch build).

In [ ]:
from IPython.display import Markdown, Image, display
for name in ('table_sev5_Standard.md', 'table_sev5_Hendrycks2020AugMix_WRN.md',
             'severity_trend.md', 'variance.md', 'deviation_report.md'):
    p = f'{OUT_ROOT}/{name}'
    if os.path.exists(p):
        display(Markdown(open(p).read()))
display(Image(f'{OUT_ROOT}/severity_trend.png'))